In [18]:
import sys
from pathlib import Path
import io
import pandas as pd

sys.path.append(str(Path.cwd().parent))

from src.storage.minio_client import read_parquet
from src.storage.minio_client import upload_bytes

In [19]:
df_silver = read_parquet(
    bucket_name="silver",
    object_name="networks_clean.parquet"
)

In [20]:
df_silver.head()

,network_id,network_name,city,country,latitude,longitude,company,system,href
0,abu-dhabi-careem-bike,Abu Dhabi Careem BIKE,Abu Dhabi,AE,24.486600,54.372800,Careem,None,/v2/networks/abu-dhabi-careem-bike
1,acces-velo-saguenay,Accès Vélo,Saguenay,CA,48.433333,-71.083333,PBSC Urban Solutions,None,/v2/networks/acces-velo-saguenay
2,aksu,Aksu,阿克苏市 (Aksu City),CN,41.166400,80.261700,阿克苏公共服务,None,/v2/networks/aksu
3,alba,Alba,Alba,IT,44.716667,8.083333,Comunicare S.r.l.,Bicincittà,/v2/networks/alba
4,albabici,AlbaBici,Albacete,ES,38.994300,-1.860200,Instituto Tecnológico de Castilla y León (ITCL),bicicard,/v2/networks/albabici


In [21]:
# Create dimensional table (1)
 
dim_city = (
    df_silver[
        ["city", "country"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_city.insert(
    0,
    "city_key",
    range(1, len(dim_city) + 1)
)

In [22]:
# Create dimensional table (2)

dim_system = (
    df_silver[
        ["system"]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

dim_system.insert(
    0,
    "system_key",
    range(1, len(dim_system) + 1)
)

In [23]:
# Creamos la tabla de hechos - fact table

fact_network = df_silver.merge(
    dim_city,
    on=["city", "country"],
    how="left"
)

fact_network = fact_network.merge(
    dim_system,
    on=["system"],
    how="left"
)

In [24]:
fact_network = fact_network[
    [
        "network_id",
        "network_name",
        "city_key",
        "system_key",
        "latitude",
        "longitude",
        "company",
        "href"
    ]
]

In [25]:
buffer = io.BytesIO()

dim_city.to_parquet(
    buffer,
    index=False
)

buffer.seek(0)

upload_bytes(
    bucket_name="gold",
    object_name="dim_city.parquet",
    data=buffer.getvalue(),
    content_type="application/octet-stream"
)

Archivo 'dim_city.parquet' subido correctamente al bucket 'gold'.


In [26]:
buffer = io.BytesIO()

dim_system.to_parquet(
    buffer,
    index=False
)

buffer.seek(0)

upload_bytes(
    bucket_name="gold",
    object_name="dim_system.parquet",
    data=buffer.getvalue(),
    content_type="application/octet-stream"
)

Archivo 'dim_system.parquet' subido correctamente al bucket 'gold'.


In [27]:
buffer = io.BytesIO()

fact_network.to_parquet(
    buffer,
    index=False
)

buffer.seek(0)

upload_bytes(
    bucket_name="gold",
    object_name="fact_network.parquet",
    data=buffer.getvalue(),
    content_type="application/octet-stream"
)

Archivo 'fact_network.parquet' subido correctamente al bucket 'gold'.
